# 1 · Talking to an LLM

⏱ about 6 minutes

In this workshop you build a chatbot that answers questions about three scientific papers
and shows you the page each answer comes from. Five short notebooks, one piece each:

1. **Chat** — talk to a language model *(this notebook)*
2. **Documents** — read PDFs and cut them into chunks
3. **Search** — find the chunks that match a question
4. **RAG** — answer from those chunks, with citations
5. **Together** — all pieces in one chatbot

![The two halves of RAG: preparing the documents once, and answering every question](img/overview.svg)

**How the notebooks drive the chatbot.** Some cells start with `%%writefile app.py`. They don't
run code; they *write* the chatbot's source file. A Chainlit server watches that file and
reloads whenever it changes. Run the cell, switch to the browser tab, and the bot has changed.

![A notebook cell writes app.py, Chainlit notices and reloads the browser](img/live-reload.svg)

Each notebook ends with **Your turn**: small changes to the chatbot that you try on your own.
Run the cells from top to bottom with **Shift+Enter**.

## Connect to the model

This reads your gateway address and key from `.env` and asks for one word. The **gateway** is our
server in Potsdam that runs the open models. It speaks the same format as OpenAI's API, so the
`openai` Python package works as the client; nothing goes to OpenAI.

In [1]:
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv()
CHAT_MODEL = os.environ["CHAT_MODEL"]
EMBED_MODEL = os.environ["EMBED_MODEL"]
llm = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])


def show(answer):
    """Show a model answer as formatted text, set off as a quote so it doesn't blend into the notebook."""
    display(Markdown("\n".join("> " + line for line in answer.splitlines())))


reply = llm.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "Say hello in one word."}],
)
show(reply.choices[0].message.content)

> Hello

An LLM call is a list of messages in, one message out. Let's ask about a detail from one of our papers.

*Kage et al. (2018)* comes from bioanalytics: flow cytometry, which counts tiny particles such as
cells as they stream past a laser. The paper tags tiny beads by how long they glow after a flash of
light, so the instrument can tell them apart. You don't need to understand it: you only judge an
answer about it.

In [2]:
QUESTION = "In Kage et al. (2018), what were the lifetime-encoded beads loaded with, and which lifetimes did they have?"

reply = llm.chat.completions.create(model=CHAT_MODEL, messages=[{"role": "user", "content": QUESTION}])
answer = reply.choices[0].message.content
open("notebook1_answer.md", "w").write(answer)  # notebook 4 checks this answer
show(answer)

> In Kage et al. (2018), the lifetime-encoded beads were loaded with **quantum dots (QDs)**.
> 
> They utilized three different types of quantum dots to create beads with distinct fluorescence lifetimes, which served as "barcode" identifiers for the beads. The lifetimes for these beads were:
> 
> *   **Short lifetime:** $\approx 10\text{ ns}$
> *   **Medium lifetime:** $\approx 25\text{ ns}$
> *   **Long lifetime:** $\approx 50\text{ ns}$
> 
> These lifetime-encoded beads were used in conjunction with Fluorescence Lifetime Imaging Microscopy (FLIM) to allow for the multiplexed detection of multiple targets by distinguishing the beads based on their temporal decay rather than just their emission color.

### Would you rely on this answer?

It sounds confident. Decide now: **yes, as it is**, **yes, after a quick check**, or **no**.
Remember your answer; notebook 4 checks this exact answer against the paper.

## Your first Chainlit app

[Chainlit](https://docs.chainlit.io) turns Python functions into a chat window. The line
`@cl.on_message` above a function tells Chainlit to call it with every message you type.

In [ ]:
%%writefile app.py
import chainlit as cl


@cl.on_message
async def on_message(message: cl.Message):
    await cl.Message(content=f"You said: {message.content}").send()

Now start the server. It keeps running in the background, also while you work through the
other notebooks, and reloads whenever `app.py` changes.

In [ ]:
import socket
import subprocess
import sys

# Start the chatbot, unless it's still running from an earlier notebook.
with socket.socket() as probe:
    running = probe.connect_ex(("127.0.0.1", 8000)) == 0
if not running:
    subprocess.Popen(
        [sys.executable, "-m", "chainlit", "run", "app.py", "--watch", "--headless"],
        stdout=open("chainlit.log", "w"),
        stderr=subprocess.STDOUT,
    )
print("Chatbot: http://localhost:8000")

### Try it: open http://localhost:8000

Say something; the bot repeats it.

> If the bot misbehaves, look at the end of the log: `!tail -20 chainlit.log`

## A real chatbot

The model has no memory, so we send the whole conversation every time. `cl.user_session` keeps it
separately for every browser tab.

With `stream=True` the answer arrives token by token (a token is a word or a piece of one), so you
watch it being typed. While one chat waits for the model, Chainlit serves the others, so the app
uses `AsyncOpenAI`: `async def` marks a function that waits, and `await` marks where it waits.
That is all you need: keep `async def`, and keep `await` in front of calls to the model and to
`.send()`. The two places are marked in the cell below.

In [ ]:
%%writefile app.py
import os

import chainlit as cl
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()
llm = AsyncOpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])

# TODO 1: give the bot a personality.
SYSTEM_PROMPT = "You are a helpful assistant. Keep your answers short."



@cl.on_chat_start
async def on_chat_start():
    cl.user_session.set("history", [{"role": "system", "content": SYSTEM_PROMPT}])


@cl.on_message
async def on_message(message: cl.Message):
    # Bonus: start a stopwatch here ...
    history = cl.user_session.get("history")  # history: the whole conversation, every time
    history.append({"role": "user", "content": message.content})

    answer = cl.Message(content="")
    stream = await llm.chat.completions.create(
        model=os.environ["CHAT_MODEL"], messages=history, stream=True  # streaming: token by token
    )
    async for part in stream:
        if part.choices and part.choices[0].delta.content:
            await answer.stream_token(part.choices[0].delta.content)
    # ... and show the time under the answer here.
    await answer.send()

    history.append({"role": "assistant", "content": answer.content})

### Try it: open http://localhost:8000

The tab reloads by itself when `app.py` changes; if it doesn't, reload it. Chat with the bot: does it remember what you said two messages ago?

## Your turn

*Bonus* exercises are only for when you finish early. When the time is up, we move on together.

Edit the `%%writefile app.py` cell above, run it again, then switch to **http://localhost:8000** and try
your change.

1. **Personality.** Change `SYSTEM_PROMPT` so the bot answers like a pirate, or only in German,
   or only in haiku. Start a new chat (the pencil icon, top left) to pick up the new prompt.

**Bonus: stopwatch.** Show under every answer how long it took. `time.perf_counter()` returns the
current time in seconds; you can add text to `answer.content` before `answer.send()`.

<details>
<summary><b>Solutions</b> (try first!)</summary>

**1** — any instruction works:
```python
SYSTEM_PROMPT = "You are a pirate. Answer every question like a pirate, in two sentences at most."
```

**Bonus** — `import time` at the top, then in `on_message`:
```python
    start = time.perf_counter()
    ...
    answer.content += f"\n\n*{time.perf_counter() - start:.1f} seconds*"
    await answer.send()
```
</details>

## Questions people ask

**Can the empty chat show example questions?** Yes, Chainlit calls them starters:
```python
@cl.set_starters
async def set_starters():
    return [
        cl.Starter(label="What is RAG?", message="Explain retrieval-augmented generation in two sentences."),
        cl.Starter(label="Tell a joke", message="Tell me a joke about scientists."),
    ]
```